# 第 14 章实验：端到端推理、导出与一致性
把合成 RGB 输入、预处理、模型输出、TorchScript 跟踪和延迟统计串成最小离线流水线。导出文件只存在于临时目录。

In [ ]:
import tempfile
import time
from pathlib import Path
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(14)
input_hw = (32, 32)
class_names = ['empty', 'object']

In [ ]:
image = np.zeros((60, 100, 3), dtype=np.float32)
image[15:48, 35:75, 0] = 0.9
image[15:48, 35:75, 1] = 0.35

def preprocess(rgb_hwc):
    assert rgb_hwc.ndim == 3 and rgb_hwc.shape[2] == 3
    tensor = torch.from_numpy(rgb_hwc).permute(2, 0, 1).unsqueeze(0)
    return F.interpolate(tensor, size=input_hw, mode='bilinear', align_corners=False)

x = preprocess(image)
print('original HWC:', image.shape, 'model input NCHW:', tuple(x.shape), x.dtype)

In [ ]:
class DeployableTinyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(3, 6, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(6, 2))
    def forward(self, x):
        return self.net(x)

model = DeployableTinyNet().eval()
with torch.inference_mode():
    logits = model(x)
    probabilities = logits.softmax(1)
result = {'class_id': int(probabilities.argmax(1)), 'score': float(probabilities.max())}
print('output logits:', tuple(logits.shape), 'result contract:', result)

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    path = Path(tmpdir) / 'tiny_model.pt'
    traced = torch.jit.trace(model, x)
    traced.save(str(path))
    loaded = torch.jit.load(str(path)).eval()
    with torch.inference_mode():
        eager_out = model(x)
        exported_out = loaded(x)
    max_error = float((eager_out - exported_out).abs().max())
    file_size = path.stat().st_size
print(f'export max abs error={max_error:.3e}, temporary file bytes={file_size}')
assert max_error < 1e-6

In [ ]:
def benchmark(module, tensor, repeats=40):
    samples_ms = []
    with torch.inference_mode():
        for _ in range(5): module(tensor)
        for _ in range(repeats):
            start = time.perf_counter(); module(tensor); samples_ms.append((time.perf_counter()-start)*1000)
    return float(np.percentile(samples_ms, 50)), float(np.percentile(samples_ms, 95))

p50, p95 = benchmark(model, x)
print(f'CPU single-sample latency: P50={p50:.3f} ms, P95={p95:.3f} ms')
print('这是教学测量，不是目标设备正式基准。')

## 破坏实验：RGB/BGR 契约不一致
部署前后颜色通道顺序必须一致。下面交换通道并比较 logits。

In [ ]:
bgr_x = preprocess(image[..., ::-1].copy())
with torch.inference_mode(): channel_error = float((model(x) - model(bgr_x)).abs().max())
print(f'RGB/BGR swap changed logits by max {channel_error:.6f}')
fig, axes = plt.subplots(1, 2, figsize=(6, 2.5))
axes[0].imshow(image); axes[0].set_title('RGB input')
axes[1].bar(class_names, probabilities[0].detach()); axes[1].set_ylim(0, 1); axes[1].set_title('output probabilities')
axes[0].axis('off'); plt.tight_layout(); plt.show()

## 记录要求
正式项目还需固定数据版本、预处理、类别表、阈值和后处理，并在真实目标设备测端到端 P50/P95、峰值内存与任务指标。